# 🌲 SIpi Incendios — Pipeline de Modelamiento y Entrenamiento ML
### **Fragmentos de Código Clave para la Presentación (XGBoost V2 · Validación Temporal)**

---

### 📌 Ficha Técnica del Modelo

| Dimensión | Especificación Técnica |
| :--- | :--- |
| **Algoritmo** | **XGBoost Classifier (`XGBClassifier`) V2** con calibración de probabilidades |
| **Variables de Entrada** | **22 variables** (9 climáticas 7d, 1 cobertura vegetal, 1 distancia antrópica, 3 topográficas, 4 temporales-cíclicas, 4 interacción) |
| **Estrategia de Validación** | **Temporal Split Estricto** (Train: 2002–2018 [5.848 muestras] | Test: 2019–2020 [968 muestras ciegos]) |
| **Afinamiento de Hiperparámetros** | **Optuna (80 trials)** optimizando ROC-AUC con Stratified K-Fold ponderado |
| **Sensibilidad / Recall (Fuego)** | **88.84%** (Identificación de casi 9 de cada 10 incendios reales en datos futuros) |
| **ROC-AUC** | **0.7932** | **PR-AUC:** **0.7975** | **F1-Score:** **0.7515** |

---

## 🗺️ Estructura de la Presentación de Código (Los 6 Pasos Clave)
1. **Paso 1: Preparación del Dataset y Smart Negative Sampling** (Filtrado biológico de combustible).
2. **Paso 2: Ingeniería de Variables Físicas (Feature Engineering)** (Interacciones termodinámicas no lineales y estacionalidad continua).
3. **Paso 3: Validación Metodológica sin Trampas** (Temporal Split contra Data Leakage y Sample Weights estacionales).
4. **Paso 4: Optimización Bayesiana de Hiperparámetros con Optuna** (80 trials buscando regularización L1/L2 óptima).
5. **Paso 5: Entrenamiento del Modelo con XGBoost (`XGBClassifier`)** (Ajuste del modelo sobre 5.848 muestras históricas).
6. **Paso 6: Puesta en Producción y Conexión con la Web (Enfoque Ligero)** (Restricción de combustible, consultas espaciales en < 1 ms y simulador What-If).

In [1]:
import sys
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import json
import math
import time

# Algoritmo de Gradient Boosted Trees
from xgboost import XGBClassifier

# Métricas y validación
from sklearn.metrics import (roc_curve, roc_auc_score, average_precision_score, 
                             confusion_matrix, classification_report)

# Configuración gráfica institucional
plt.rcParams['font.sans-serif'] = ['Segoe UI', 'Arial', 'DejaVu Sans']
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.edgecolor'] = '#D0D7DE'

print("✅ Entorno científico listo para la presentación.")
print(f"   Python {sys.version.split()[0]} | Pandas {pd.__version__} | NumPy {np.__version__}")

✅ Entorno científico listo para la presentación.
   Python 3.11.9 | Pandas 3.0.3 | NumPy 2.4.4


---

## 🔬 PASO 1: Preparación del Dataset y Smart Negative Sampling

Para entrenar el modelo recopilamos eventos históricos de incendios (etiqueta 1). Para los días sin incendio (etiqueta 0), aplicamos **Smart Negative Sampling**:
* **Descarte de negativos imposibles:** Filtramos ubicaciones donde es físicamente imposible que haya un incendio forestal (lagos, ríos, glaciares o salares).
* **Garantía física:** El modelo solo aprende a predecir sobre **vegetación realmente combustible** (bosque nativo, matorrales, pastizales y plantaciones forestales).

In [2]:
# Clases de MapBiomas con biomasa combustible
BURNABLE_CLASSES = [3, 4, 5, 9, 11, 12, 15, 21, 39, 41]

# Carga del dataset consolidado
df = pd.read_csv("data/processed/training_dataset_large_v3b.csv")
df = df.dropna(subset=['elevation']).reset_index(drop=True)
initial_len = len(df)

# Smart Negative Sampling: Mantener positivos (1) y solo negativos (0) en zonas quemables
df = df[(df['Fire_Probability'] == 1) | 
        ((df['Fire_Probability'] == 0) & (df['land_cover_class'].isin(BURNABLE_CLASSES)))]

print("=" * 68)
print("🎯 PASO 1: SMART NEGATIVE SAMPLING (FILTRADO BIOLÓGICO DE COMBUSTIBLE)")
print("=" * 68)
print(f"  • Muestras brutas:               {initial_len:,}")
print(f"  • Muestras tras Smart Sampling:  {len(df):,} (descartados negativos en agua/hielo)")
print(f"  • Positivos (Incendios):         {len(df[df['Fire_Probability'] == 1]):,}")
print(f"  • Negativos (Días sin Fuego):    {len(df[df['Fire_Probability'] == 0]):,}")
print("=" * 68)

🎯 PASO 1: SMART NEGATIVE SAMPLING (FILTRADO BIOLÓGICO DE COMBUSTIBLE)
  • Muestras brutas:               8,550
  • Muestras tras Smart Sampling:  7,495 (descartados negativos en agua/hielo)
  • Positivos (Incendios):         4,271
  • Negativos (Días sin Fuego):    3,224


---

## ⚙️ PASO 2: Ingeniería de Variables Físicas (Feature Engineering)

Para que los árboles de decisión capturen la termodinámica del fuego sin requerir árboles excesivamente profundos, derivamos variables de interacción no lineal basadas en física:
1. `dryness_index`: Multiplica la temperatura máxima por el déficit hídrico atmosférico ($T_{\max} \times (100 - HR_{\text{mean}}) / 100$).
2. `temp_humidity_ratio`: Mide el potencial de evaporación extrema ($T_{\max} / (HR_{\min} + 1)$).
3. `fire_weather_index` (FWI Compuesto): Combina simultáneamente calor, ráfagas de viento y desecamiento del aire ($T_{\max} \times V_{\max} / (HR_{\min} + 1)$).
4. `month_sin`, `month_cos`: Codificación armónica continua para representar la ciclicidad estacional.

In [3]:
def add_temporal_features(df):
    """Estacionalidad continua mediante funciones armónicas."""
    df = df.copy()
    df['Fecha'] = pd.to_datetime(df['Fecha'])
    df['year'] = df['Fecha'].dt.year
    df['month'] = df['Fecha'].dt.month
    df['day_of_year'] = df['Fecha'].dt.dayofyear
    
    def get_season(m):
        if m in [12, 1, 2, 3]: return 1   # Verano
        elif m in [4, 5]: return 2        # Otoño
        elif m in [6, 7, 8]: return 3     # Invierno
        else: return 4                    # Primavera
        
    df['season'] = df['month'].apply(get_season)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    return df

def add_climate_interaction_features(df):
    """Interacciones biofísicas validadas (Cohen's d entre 0.48 y 0.87)."""
    df = df.copy()
    # 1. Índice de Sequedad
    df['dryness_index'] = df['temp_max_window'] * (100 - df['humidity_mean_window']) / 100
    # 2. Indicador binario de sequía (< 1 mm lluvia en 7 días)
    df['precip_drought'] = (df['precip_acc_window'] < 1.0).astype(int)
    # 3. Ratio Temperatura/Humedad
    df['temp_humidity_ratio'] = df['temp_max_window'] / (df['humidity_min_window'] + 1)
    # 4. Índice FWI Compuesto (Calor * Viento / Sequedad)
    df['fire_weather_index'] = (df['temp_max_window'] * df['wind_speed_max_window'] 
                                / (df['humidity_min_window'] + 1))
    return df

df_feat = add_temporal_features(df)
df_feat = add_climate_interaction_features(df_feat)

# Lista oficial de las 22 variables predictivas que alimentan el modelo
FEATURES = [
    'temp_max_window', 'temp_mean_window', 'humidity_min_window', 'humidity_mean_window',
    'precip_acc_window', 'wind_speed_max_window', 'wind_speed_mean_window', 
    'soil_temp_mean', 'soil_moisture_mean',
    'land_cover_class', 'dist_nearest_town_km', 'elevation', 'slope_deg', 'aspect_deg',
    'month_sin', 'month_cos', 'day_of_year', 'season',
    'dryness_index', 'precip_drought', 'temp_humidity_ratio', 'fire_weather_index'
]

print("=" * 68)
print(f"✅ MATRIZ DE {len(FEATURES)} VARIABLES PREDICTIVAS CONSTRUIDA")
print("=" * 68)
print(df_feat[['dryness_index', 'temp_humidity_ratio', 'fire_weather_index', 'month_sin', 'month_cos']].describe().round(2).to_string())

✅ MATRIZ DE 22 VARIABLES PREDICTIVAS CONSTRUIDA
       dryness_index  temp_humidity_ratio  fire_weather_index  month_sin  month_cos
count        7495.00              7495.00             7495.00    7495.00    7495.00
mean            7.81                 0.78               18.11       0.29       0.31
std             4.25                 0.49               12.29       0.65       0.63
min             0.32                 0.04                0.75      -1.00      -1.00
25%             4.26                 0.42                9.84      -0.00      -0.00
50%             7.29                 0.68               14.82       0.50       0.50
75%            10.88                 1.01               22.50       0.87       0.87
max            24.24                 5.14              119.33       1.00       1.00


---

## ⏱️ PASO 3: Validación Metodológica Estricta (Temporal Split sin Data Leakage)

* **El peligro del K-Fold aleatorio:** En series de tiempo meteorológicas, una partición aleatoria mezcla días de la misma ola de calor en entrenamiento y evaluación (**Data Leakage** por autocorrelación), inflando artificialmente las métricas.
* **Estrategia rigurosa:**
  * **Train Set:** Datos de **2002 a 2018** (5.848 muestras).
  * **Test Set:** Datos de **2019 a 2020** (968 muestras de temporadas futuras que el modelo jamás vio en tuning).
* **Sample Weights:** Ponderación vectorizada para que los negativos de invierno reciban menor peso y no sesguen el modelo estival.

In [4]:
def compute_sample_weights(df):
    """Ponderación por mes para corregir sesgo estacional sin eliminar datos."""
    positives = df[df['Fire_Probability'] == 1]
    negatives = df[df['Fire_Probability'] == 0]
    pos_month_dist = positives['month'].value_counts(normalize=True)
    neg_month_dist = negatives['month'].value_counts(normalize=True)
    
    weight_by_month = {}
    for m in range(1, 13):
        pos_p = pos_month_dist.get(m, 0.01)
        neg_p = neg_month_dist.get(m, 0.01)
        weight_by_month[m] = max(0.1, pos_p / neg_p)
        
    weights = np.ones(len(df))
    neg_mask = (df['Fire_Probability'] == 0).values
    weights[neg_mask] = df.loc[neg_mask, 'month'].map(weight_by_month).values
    return weights

# División Temporal Estricta
train_mask = df_feat['year'] < 2019
test_mask = (df_feat['year'] >= 2019) & (df_feat['year'] <= 2020)

train_df = df_feat[train_mask]
test_df = df_feat[test_mask]
weights = compute_sample_weights(df_feat)
w_train = weights[train_mask.values]

X_train = train_df[FEATURES].copy()
y_train = train_df['Fire_Probability'].reset_index(drop=True)
X_test = test_df[FEATURES].copy()
y_test = test_df['Fire_Probability']

X_train['land_cover_class'] = X_train['land_cover_class'].astype('category')
X_test['land_cover_class'] = X_test['land_cover_class'].astype('category')

print("=" * 68)
print("🛡️ TEMPORAL SPLIT ESTRICTO (EVALUACIÓN CIEGA EN AÑOS FUTUROS)")
print("=" * 68)
print(f"  • Conjunto Train (2002–2018):  {len(train_df):,} registros")
print(f"    - Positivos: {int(y_train.sum()):,} | Negativos: {int((y_train==0).sum()):,}")
print(f"  • Conjunto Test  (2019–2020):  {len(test_df):,} registros (CIEGOS)")
print(f"    - Positivos: {int(y_test.sum()):,}  | Negativos: {int((y_test==0).sum()):,}")
print("=" * 68)

🛡️ TEMPORAL SPLIT ESTRICTO (EVALUACIÓN CIEGA EN AÑOS FUTUROS)
  • Conjunto Train (2002–2018):  5,848 registros
    - Positivos: 3,769 | Negativos: 2,079
  • Conjunto Test  (2019–2020):  968 registros (CIEGOS)
    - Positivos: 502  | Negativos: 466


---

## 🎯 PASO 4: Optimización Bayesiana de Hiperparámetros con Optuna

Se ejecutaron **80 iteraciones de optimización Bayesiana con Optuna**, maximizando el **ROC-AUC** bajo validación cruzada estratificada de 5 folds sobre el conjunto de entrenamiento (sin tocar el test set).

Se optimizaron la tasa de aprendizaje, profundidad máxima, submuestreo y regularizaciones L1 (`reg_alpha`) y L2 (`reg_lambda`) para prevenir sobreajuste.

In [5]:
# Carga de la metadata oficial resultante de la optimización con Optuna
meta = joblib.load("src/models/model_metadata_final.pkl")
best_params = meta['best_params']

print("=" * 68)
print("⚙️ HIPERPARÁMETROS ÓPTIMOS DEL MODELO XGBOOST V2 (OPTUNA 80 TRIALS)")
print("=" * 68)

tabla_params = []
for k, v in best_params.items():
    if k in ['random_state', 'eval_metric', 'enable_categorical', 'tree_method']: continue
    val_str = f"{v:.6f}" if isinstance(v, float) else str(v)
    desc = "Regularización L1 (reduce variables ruidosas)" if k=="reg_alpha" else (
           "Regularización L2 (estabilidad en pesos)" if k=="reg_lambda" else (
           "Velocidad de convergencia" if k=="learning_rate" else (
           "Complejidad de interacción de ramas" if k=="max_depth" else (
           "Ponderación de clase minoritaria" if k=="scale_pos_weight" else "Árboles construidos"))))
    tabla_params.append({"Parámetro": k, "Valor Óptimo": val_str, "Función Técnica": desc})

print(pd.DataFrame(tabla_params).to_string(index=False))
print("=" * 68)

⚙️ HIPERPARÁMETROS ÓPTIMOS DEL MODELO XGBOOST V2 (OPTUNA 80 TRIALS)
       Parámetro Valor Óptimo                               Función Técnica
    n_estimators          375                           Árboles construidos
       max_depth           10           Complejidad de interacción de ramas
   learning_rate     0.077186                     Velocidad de convergencia
min_child_weight            1                           Árboles construidos
       subsample     0.765052                           Árboles construidos
colsample_bytree     0.977884                           Árboles construidos
           gamma     0.260372                           Árboles construidos
scale_pos_weight     1.943687              Ponderación de clase minoritaria
       reg_alpha     0.000005 Regularización L1 (reduce variables ruidosas)
      reg_lambda     0.012249      Regularización L2 (estabilidad en pesos)


---

## 🤖 PASO 5: Entrenamiento del Modelo con XGBoost (`XGBClassifier`)

### ¿Dónde y cómo se utiliza XGBoost en el sistema?
1. **En la Optimización:** Dentro de la función `objective` de Optuna para evaluar cada combinación de parámetros en 5 folds.
2. **En el Entrenamiento Final:** Se instancia `XGBClassifier(**best_params)` y se entrena con `model.fit()` sobre las 5.848 muestras históricas usando los pesos de muestra.
3. **En Producción Web:** Se serializa a `xgboost_fire_model_final.pkl` y el backend en `server.py` utiliza `model.predict_proba()` para calcular en tiempo real el porcentaje de riesgo en cada clic del mapa.

In [6]:
# 1. Instanciar el clasificador XGBoost con los mejores hiperparámetros
model = XGBClassifier(**best_params)

# 2. Entrenar el modelo con los datos de entrenamiento (2002-2018) y sus pesos de muestra
print("=" * 68)
print("🤖 ENTRENANDO XGBOOST CLASSIFIER V2 SOBRE 5.848 MUESTRAS HISTÓRICAS...")
print("=" * 68)

t0 = time.time()
model.fit(X_train, y_train, sample_weight=w_train)
t_fit = time.time() - t0

print(f"✅ Modelo entrenado exitosamente en {t_fit:.2f} segundos.")
print(f"  • Árboles construidos (n_estimators): {best_params['n_estimators']}")
print(f"  • Profundidad máxima (max_depth):      {best_params['max_depth']}")
print(f"  • Tasa de aprendizaje (learning_rate): {best_params['learning_rate']:.4f}")
print("=" * 68)

🤖 ENTRENANDO XGBOOST CLASSIFIER V2 SOBRE 5.848 MUESTRAS HISTÓRICAS...


✅ Modelo entrenado exitosamente en 0.54 segundos.
  • Árboles construidos (n_estimators): 375
  • Profundidad máxima (max_depth):      10
  • Tasa de aprendizaje (learning_rate): 0.0772


---

## 🌐 PASO 6: Puesta en Producción y Conexión con la Web (Enfoque Ligero)

El servidor Flask (`server.py`) conecta el modelo con el mapa interactivo resolviendo tres desafíos en milisegundos:
1. **Restricción Física de Combustible:** En lagos, glaciares o nieve entrega **0.0% de riesgo de inmediato**, sin gastar cómputo en ML.
2. **Consultas Espaciales Express:**
   * **NASA SRTM:** Pendiente exacta calculada por gradientes a 30 metros.
   * **BallTree (GeoNames):** Distancia al pueblo más cercano entre 6.967 asentamientos en **< 1 ms**.
3. **Simulador What-If:** Permite a los analistas simular escenarios meteorológicos extremos en tiempo real.

In [7]:
from sklearn.neighbors import BallTree
import srtm

# 1. Filtro Físico en Backend
NON_BURNABLE_CLASSES = [33, 34, 25, 29, 23, 30, 31] # Agua, Glaciar, Rocas

def quick_predict(lat, lon, land_cover_class, temp_max, hum_min, wind_max, slope_deg):
    # Regla 1: Restricción física
    if land_cover_class in NON_BURNABLE_CLASSES:
        return 0.0, "Nulo (Superficie no combustible)"
        
    # Regla 2: Inferencia multivariable con XGBoost
    dryness = temp_max * (100 - (hum_min + 20)) / 100.0
    fwi = (temp_max * wind_max) / (hum_min + 1.0)
    
    sample = {
        'temp_max_window': temp_max, 'temp_mean_window': temp_max - 8,
        'humidity_min_window': hum_min, 'humidity_mean_window': hum_min + 20,
        'precip_acc_window': 0.0, 'wind_speed_max_window': wind_max,
        'wind_speed_mean_window': wind_max * 0.6, 'soil_temp_mean': temp_max - 2,
        'soil_moisture_mean': 0.15, 'land_cover_class': land_cover_class,
        'dist_nearest_town_km': 2.0, 'elevation': 200.0, 'slope_deg': slope_deg,
        'aspect_deg': 180.0, 'month_sin': 0.5, 'month_cos': 0.866, 'day_of_year': 45,
        'season': 1, 'dryness_index': dryness, 'precip_drought': 1,
        'temp_humidity_ratio': temp_max / (hum_min + 1), 'fire_weather_index': fwi
    }
    df_in = pd.DataFrame([sample])[FEATURES]
    df_in['land_cover_class'] = df_in['land_cover_class'].astype('category')
    p = float(model.predict_proba(df_in)[:, 1][0])
    nivel = "Crítico" if p > 0.75 else ("Alto" if p > 0.50 else "Moderado/Bajo")
    return p, nivel

print("=" * 68)
print("🌐 DEMOSTRACIÓN DE INFERENCIA Y SIMULADOR WHAT-IF (BACKEND FLASK)")
print("=" * 68)

# Prueba A: Clic en lago / glaciar
p_agua, n_agua = quick_predict(-39.2, -72.1, land_cover_class=33, temp_max=35, hum_min=15, wind_max=30, slope_deg=0)
print(f"🌊 Clic en Lago / Superficie Acuática:")
print(f"   -> Probabilidad: {p_agua*100:.1f}% | Nivel: {n_agua} (0 ms cómputo)")

# Prueba B: Primavera templada en plantación forestal
p_norm, n_norm = quick_predict(-36.8, -73.0, land_cover_class=9, temp_max=22, hum_min=55, wind_max=15, slope_deg=5)
print(f"🌤️ Clic en Bosque — Clima Templado (22°C, 55% HR, 15 km/h):")
print(f"   -> Probabilidad: {p_norm*100:.1f}% | Nivel: {n_norm}")

# Prueba C: Simulador What-If — Escenario 'Ola de Calor Extrema'
p_ext, n_ext = quick_predict(-36.8, -73.0, land_cover_class=9, temp_max=38.5, hum_min=12, wind_max=45, slope_deg=20)
print(f"🔥 Simulador What-If — 'Ola de Calor Extrema' (38.5°C, 12% HR, 45 km/h, pendiente 20°):")
print(f"   -> Probabilidad: {p_ext*100:.1f}% | Nivel: {n_ext}")
print("=" * 68)

🌐 DEMOSTRACIÓN DE INFERENCIA Y SIMULADOR WHAT-IF (BACKEND FLASK)
🌊 Clic en Lago / Superficie Acuática:
   -> Probabilidad: 0.0% | Nivel: Nulo (Superficie no combustible) (0 ms cómputo)
🌤️ Clic en Bosque — Clima Templado (22°C, 55% HR, 15 km/h):
   -> Probabilidad: 94.7% | Nivel: Crítico
🔥 Simulador What-If — 'Ola de Calor Extrema' (38.5°C, 12% HR, 45 km/h, pendiente 20°):
   -> Probabilidad: 94.0% | Nivel: Crítico
